In [ ]:
"""
✅ FINAL VERSION WITH FIXED SCHEDULER
Swin Small Face Forensics - Recursive Folder Loading + Correct Scheduler
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
from datetime import datetime
import json

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    BATCH_SIZE = 32
    EPOCHS = 12
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 5e-5
    EARLY_STOPPING_PATIENCE = 8
    
    # ✅ FIXED SCHEDULER SETTINGS
    SCHEDULER_TYPE = 'warmup_step'  # Options: 'warmup_step', 'warmup_decay', 'cosine_warmup'
    WARMUP_EPOCHS = 2
    STEP_SIZE = 8  # For step scheduler
    GAMMA = 0.5    # Decay factor
    
    MODEL_NAME = 'swin_small'
    NUM_CLASSES = 2
    PRETRAINED = True
    
    IMAGE_SIZE = 256
    TRAIN_VAL_SPLIT = 0.85
    NUM_WORKERS = 2
    
    USE_MIXED_PRECISION = True
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    INPUT_DIR = '/kaggle/input/datasets/shreyanshpatel1/130k-real-vs-fake-face/images'
    OUTPUT_DIR = '/kaggle/working'
    
    print(f"""
    ╔═══════════════════════════════════════════════════════╗
    ║     ✅ FIXED SCHEDULER - swin_face_forensics_FINAL    ║
    ║                                                       ║
    ║  Device: {str(torch.cuda.get_device_name(0)[:25]) if torch.cuda.is_available() else 'CPU':40} ║
    ║  Batch Size: 32                                      ║
    ║  Learning Rate: 1e-4                                 ║
    ║  Scheduler: Warmup + Step (FIXED)                    ║
    ║  Warmup Epochs: 2                                    ║
    ║  Step Size: 8                                        ║
    ║  Gamma: 0.5                                          ║
    ╚═══════════════════════════════════════════════════════╝
    """)

config = Config()

# ============================================================================
# FIXED SCHEDULER IMPLEMENTATIONS
# ============================================================================

def create_warmup_step_scheduler(optimizer, warmup_epochs=2, step_size=8, gamma=0.5):
    """
    ✅ FIXED: Linear warmup + step decay
    
    Learning rate progression:
    ├─ Epoch 1: 0.5x (warmup start)
    ├─ Epoch 2: 1.0x (warmup end)
    ├─ Epochs 3-10: 1.0x (constant)
    ├─ Epochs 11-18: 0.5x (step decay)
    └─ Epochs 19-26: 0.25x (step decay)
    """
    
    def lr_lambda(epoch):
        epoch = epoch + 1  # ✅ START FROM EPOCH 1, NOT 0
        
        if epoch <= warmup_epochs:
            # Warmup phase: gradually increase LR
            return float(epoch) / float(max(1, warmup_epochs))
        else:
            # Step decay after warmup
            return gamma ** ((epoch - warmup_epochs) // step_size)
    
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def create_warmup_decay_scheduler(optimizer, warmup_epochs=2, total_epochs=30):
    """
    ✅ FIXED: Linear warmup + linear decay
    
    Learning rate progression:
    ├─ Epoch 1: 0.5x (warmup)
    ├─ Epoch 2: 1.0x (warmup end)
    ├─ Epoch 15: 0.5x (decay middle)
    └─ Epoch 30: ~0.0x (decay end)
    """
    
    def lr_lambda(epoch):
        epoch = epoch + 1  # ✅ START FROM EPOCH 1
        
        if epoch <= warmup_epochs:
            # Warmup phase
            return float(epoch) / float(max(1, warmup_epochs))
        else:
            # Decay phase
            remaining_epochs = total_epochs - epoch + 1
            total_decay_epochs = total_epochs - warmup_epochs
            return max(0.0, float(remaining_epochs) / float(max(1, total_decay_epochs)))
    
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def create_cosine_warmup_scheduler(optimizer, warmup_epochs=2, total_epochs=30):
    """
    ✅ FIXED: Linear warmup + cosine decay
    
    Learning rate progression:
    ├─ Epochs 1-2: Linear warmup (0.5 → 1.0)
    ├─ Epochs 2-30: Cosine decay (1.0 → 0.0)
    """
    import math
    
    def lr_lambda(epoch):
        epoch = epoch + 1  # ✅ START FROM EPOCH 1
        
        if epoch <= warmup_epochs:
            # Warmup
            return float(epoch) / float(max(1, warmup_epochs))
        else:
            # Cosine decay
            progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
            return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ============================================================================
# DATASET CLASS
# ============================================================================

class FaceForensicsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
        print(f"📊 Dataset: {len(image_paths)} images")
        print(f"   Real: {sum(1 for l in labels if l == 0)}")
        print(f"   Fake: {sum(1 for l in labels if l == 1)}")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        
        try:
            img = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                img = self.transform(img)
            label = torch.tensor(self.labels[idx], dtype=torch.long)
            return img, label
        except Exception as e:
            return torch.zeros(3, config.IMAGE_SIZE, config.IMAGE_SIZE), torch.tensor(0)

# ============================================================================
# DATA PREPARATION - RECURSIVE LOADING
# ============================================================================

def prepare_dataset_recursive():
    """
    ✅ Recursively loads images from ALL subdirectories
    Handles nested /fake/ subfolder structure
    """
    print("\n" + "="*70)
    print("📥 LOADING DATASET (RECURSIVE - HANDLES SUBFOLDERS)")
    print("="*70)
    
    image_paths = []
    labels = []
    
    dataset_dir = config.INPUT_DIR
    
    if not os.path.exists(dataset_dir):
        print(f"❌ Dataset not found at {dataset_dir}")
        print(f"Available: {os.listdir('/kaggle/input')}")
        return None
    
    print(f"✓ Dataset found at: {dataset_dir}")
    
    # Load REAL faces (flat structure)
    real_dir = os.path.join(dataset_dir, 'real')
    real_count = 0
    
    if os.path.exists(real_dir):
        print(f"\n📂 Loading /real/ folder (flat structure)...")
        for img_file in os.listdir(real_dir):
            img_path = os.path.join(real_dir, img_file)
            if os.path.isfile(img_path) and img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                image_paths.append(img_path)
                labels.append(0)  # Real
                real_count += 1
        print(f"   ✓ Loaded {real_count} real faces")
    else:
        print(f"⚠️  /real/ folder not found!")
    
    # Load FAKE faces (RECURSIVE - handles subfolders)
    fake_dir = os.path.join(dataset_dir, 'fake')
    fake_count = 0
    subfolder_names = []
    
    if os.path.exists(fake_dir):
        print(f"\n📂 Loading /fake/ folder (recursive search)...")
        
        # Load images directly in /fake/
        for img_file in os.listdir(fake_dir):
            img_path = os.path.join(fake_dir, img_file)
            if os.path.isfile(img_path) and img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                image_paths.append(img_path)
                labels.append(1)  # Fake
                fake_count += 1
        
        if fake_count > 0:
            print(f"   ✓ Found {fake_count} images directly in /fake/")
        
        # Load images from subfolders
        for item in os.listdir(fake_dir):
            item_path = os.path.join(fake_dir, item)
            
            if os.path.isdir(item_path):
                subfolder_names.append(item)
                subfolder_count = 0
                
                # Recursively search subdirectory
                for root, dirs, files in os.walk(item_path):
                    for img_file in files:
                        if img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                            full_path = os.path.join(root, img_file)
                            image_paths.append(full_path)
                            labels.append(1)  # Fake
                            subfolder_count += 1
                            fake_count += 1
                
                print(f"   ✓ Found {subfolder_count} images in /fake/{item}/")
        
        print(f"   ✓ Total fake faces: {fake_count}")
    else:
        print(f"⚠️  /fake/ folder not found!")
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"📊 DATASET LOADED SUCCESSFULLY")
    print(f"{'='*70}")
    print(f"Real faces:    {real_count:,}")
    print(f"Fake faces:    {fake_count:,}")
    print(f"Total:         {real_count + fake_count:,}")
    
    if len(subfolder_names) > 0:
        print(f"\nFake subfolders found: {', '.join(subfolder_names)}")
    
    ratio = real_count / max(fake_count, 1)
    print(f"Class ratio:   {ratio:.2f}:1 (Real:Fake)")
    
    # Check if looks reasonable
    if real_count < 50000 or fake_count < 50000:
        print(f"\n⚠️  WARNING: Dataset smaller than expected!")
        print(f"   Expected: ~70k real, ~63k fake")
        print(f"   Got:      {real_count:,} real, {fake_count:,} fake")
    else:
        print(f"\n✅ Dataset looks good!")
    
    # Shuffle
    indices = np.random.permutation(len(image_paths))
    image_paths = [image_paths[i] for i in indices]
    labels = [labels[i] for i in indices]
    
    return image_paths, labels

def create_data_loaders(image_paths, labels):
    """Create train/val data loaders"""
    
    print("\n" + "="*70)
    print("🔄 CREATING DATA LOADERS")
    print("="*70)
    
    train_transform = transforms.Compose([
        transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Split data
    n_train = int(len(image_paths) * config.TRAIN_VAL_SPLIT)
    train_paths = image_paths[:n_train]
    train_labels = labels[:n_train]
    val_paths = image_paths[n_train:]
    val_labels = labels[n_train:]
    
    print(f"\n📊 Data split:")
    print(f"   Train: {len(train_paths)} images ({config.TRAIN_VAL_SPLIT*100:.0f}%)")
    print(f"   Val:   {len(val_paths)} images ({(1-config.TRAIN_VAL_SPLIT)*100:.0f}%)")
    
    # Create datasets
    train_dataset = FaceForensicsDataset(train_paths, train_labels, train_transform)
    val_dataset = FaceForensicsDataset(val_paths, val_labels, val_transform)
    
    # Create loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True,
        drop_last=False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    print(f"\n✓ Data loaders created")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Val batches: {len(val_loader)}")
    
    return train_loader, val_loader

# ============================================================================
# MODEL
# ============================================================================

class SwinFaceForensicClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(SwinFaceForensicClassifier, self).__init__()
        
        self.swin = models.swin_s(pretrained=pretrained)
        num_features = self.swin.head.in_features
        
        self.swin.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_features, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.swin(x)

def load_model():
    print("\n🧠 Loading Swin Small model...")
    model = SwinFaceForensicClassifier(num_classes=2, pretrained=True)
    model = model.to(config.DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✓ Model loaded with {total_params:,} parameters")
    return model

# ============================================================================
# TRAINING ENGINE
# ============================================================================

class FaceForensicsTrainer:
    def __init__(self, model, train_loader, val_loader, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
        
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        
        # ✅ FIXED SCHEDULER
        if config.SCHEDULER_TYPE == 'warmup_step':
            self.scheduler = create_warmup_step_scheduler(
                self.optimizer,
                warmup_epochs=config.WARMUP_EPOCHS,
                step_size=config.STEP_SIZE,
                gamma=config.GAMMA
            )
        elif config.SCHEDULER_TYPE == 'warmup_decay':
            self.scheduler = create_warmup_decay_scheduler(
                self.optimizer,
                warmup_epochs=config.WARMUP_EPOCHS,
                total_epochs=config.EPOCHS
            )
        elif config.SCHEDULER_TYPE == 'cosine_warmup':
            self.scheduler = create_cosine_warmup_scheduler(
                self.optimizer,
                warmup_epochs=config.WARMUP_EPOCHS,
                total_epochs=config.EPOCHS
            )
        
        self.scaler = torch.cuda.amp.GradScaler() if config.USE_MIXED_PRECISION else None
        
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': [],
            'best_val_acc': 0,
            'best_epoch': 0
        }
    
    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for images, labels in tqdm(self.train_loader, desc=f'Epoch {epoch+1} Train'):
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)
            
            if config.USE_MIXED_PRECISION:
                with torch.cuda.amp.autocast():
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                
                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
        
        return total_loss / len(self.train_loader), correct / total
    
    def validate(self, epoch):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(self.val_loader, desc=f'Epoch {epoch+1} Val'):
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)
        
        return total_loss / len(self.val_loader), correct / total
    
    def train(self, epochs=config.EPOCHS):
        print("\n" + "="*70)
        print("🚀 STARTING TRAINING (WITH FIXED SCHEDULER)")
        print("="*70)
        
        patience_counter = 0
        
        for epoch in range(epochs):
            print(f"\n{'─'*70}")
            print(f"Epoch {epoch+1}/{epochs}")
            print(f"{'─'*70}")
            
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate(epoch)
            
            current_lr = self.optimizer.param_groups[0]['lr']
            self.scheduler.step()
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            
            print(f"\n📊 Metrics:")
            print(f"   Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
            print(f"   Val Loss:   {val_loss:.4f} | Acc: {val_acc:.4f}")
            print(f"   LR: {current_lr:.2e}")
            
            if val_acc > self.history['best_val_acc']:
                self.history['best_val_acc'] = val_acc
                self.history['best_epoch'] = epoch + 1
                patience_counter = 0
                
                torch.save(
                    self.model.state_dict(),
                    os.path.join(config.OUTPUT_DIR, 'swin_face_forensics_best.pth')
                )
                print(f"✅ Best model saved! Val Acc: {val_acc:.4f}")
            else:
                patience_counter += 1
                print(f"⚠️  No improvement ({patience_counter}/{config.EARLY_STOPPING_PATIENCE})")
                
                if patience_counter >= config.EARLY_STOPPING_PATIENCE:
                    print(f"\n🛑 Early stopping at epoch {epoch+1}")
                    break
        
        print(f"\n{'='*70}")
        print(f"✅ Training complete!")
        print(f"{'='*70}")
        print(f"   Best epoch: {self.history['best_epoch']}")
        print(f"   Best val accuracy: {self.history['best_val_acc']:.4f}")
        
        return self.history

# ============================================================================
# EVALUATION
# ============================================================================

def evaluate_model(model, val_loader, device):
    print("\n" + "="*70)
    print("📊 FINAL EVALUATION")
    print("="*70)
    
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    print(f"\n📈 METRICS")
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Real', 'Fake'],
                yticklabels=['Real', 'Fake'],
                cbar_kws={'label': 'Count'})
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_title('Confusion Matrix - Face Forensics Detection', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'confusion_matrix.png'), dpi=100, bbox_inches='tight')
    print(f"\n✓ Confusion matrix saved")
    plt.show()
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# ============================================================================
# MAIN
# ============================================================================

def main():
    print("╔" + "="*68 + "╗")
    print("║" + "✅ FINAL VERSION - FIXED SCHEDULER".center(68) + "║")
    print("║" + "130k Real vs Fake Face Detection for KYC".center(68) + "║")
    print("╚" + "="*68 + "╝")
    
    result = prepare_dataset_recursive()
    if result is None:
        print("❌ Failed to load dataset")
        return
    
    image_paths, labels = result
    train_loader, val_loader = create_data_loaders(image_paths, labels)
    
    model = load_model()
    trainer = FaceForensicsTrainer(model, train_loader, val_loader, config.DEVICE)
    history = trainer.train(epochs=config.EPOCHS)
    
    model.load_state_dict(torch.load(os.path.join(config.OUTPUT_DIR, 'swin_face_forensics_best.pth')))
    metrics = evaluate_model(model, val_loader, config.DEVICE)
    
    torch.save(model.state_dict(), os.path.join(config.OUTPUT_DIR, 'swin_face_forensics_final.pth'))
    
    print("\n✅ Training complete! Models saved.")
    print(f"\n📁 Output files:")
    print(f"   - swin_face_forensics_best.pth")
    print(f"   - swin_face_forensics_final.pth")
    print(f"   - confusion_matrix.png")

if __name__ == "__main__":
    main()